# Setup & Model Loading

In [ ]:
import torch
from PIL import Image
import requests
import os
from src.model import SiglipBgeAligner
from transformers import AutoProcessor, AutoTokenizer, AutoModel

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model = SiglipBgeAligner().to(device)
model.load_state_dict(torch.load("model_checkpoint.pt", map_location=device))
model.eval()

In [ ]:
processor = AutoProcessor.from_pretrained(
    "google/siglip2-so400m-patch16-naflex"
)
bge_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
bge_model = AutoModel.from_pretrained("BAAI/bge-m3").to(device)
bge_model.eval()

In [ ]:
def load_image(url_or_path):
    """Loads an image from a URL or local path."""
    try:
        if url_or_path.startswith("http"):
            return Image.open(requests.get(url_or_path, stream=True).raw)
        return Image.open(url_or_path).convert("RGB")
    except Exception:
        return Image.new("RGB", (224, 224), color="red")

def get_image_embedding(images):
    """Encodes images using the fine-tuned SigLip2 model."""
    inputs = processor(images=images, return_tensors="pt", padding=True)
    vision_inputs = {k: v.to(device) for k, v in inputs.items()}
    if "pixel_attention_mask" in vision_inputs:
        vision_inputs["attention_mask"] = vision_inputs.pop(
            "pixel_attention_mask"
        )
    with torch.no_grad():
        return model(**vision_inputs)

In [ ]:
url_target = "https://www.armytimes.com/resizer/v2/OKUTW5AYJ5GR5PVLV7F2WBMQSE.jpg"
url_d1 = "https://images.unsplash.com/photo-1543466835-00a7907e9de1?w=500"
url_d2 = "https://images.unsplash.com/photo-1477959858617-67f85cf4f1df?w=500"

image_gallery = [load_image(url_target), load_image(url_d1), load_image(url_d2)]
image_names = ["US Army Photo (Target)", "Dog in a park", "Futuristic city"]

text_propositions = [
    "US Army soldiers conducting a tactical training exercise outdoors.",
    "Des soldats de l'armée américaine effectuant un exercice tactique.",
    "US-Armeesoldaten bei einer taktischen Trainingsübung im Freien.",
    "A group of firefighters working hard during an emergency.",
    "Ein glücklicher Hund, der draußen im Gras spielt.",
    "An abstract painting of a futuristic city skyline."
]

# Calculate Embeddings
text_inputs = bge_tokenizer(
    text_propositions, padding=True, truncation=True, return_tensors="pt"
).to(device)

with torch.no_grad():
    text_out = bge_model(**text_inputs)
    text_embeddings = text_out.last_hidden_state[:, 0, :]
    text_embeddings = torch.nn.functional.normalize(text_embeddings, p=2, dim=-1)

image_embeddings = get_image_embedding(image_gallery)

In [ ]:
# Test 1: Image to Multi-Text
print(f"{'='*50}\nTEST 1: IMAGE -> MULTI-TEXT ALIGNMENT\n{'='*50}")
scores = (image_embeddings[0:1] @ text_embeddings.T).squeeze().cpu().numpy()

for i, text in enumerate(text_propositions):
    status = "High" if i < 3 else "Low"
    print(f"Score: {scores[i]:.4f} [{status}] -> {text[:50]}...")

# Test 2: Text to Multi-Image
print(f"\n{'='*50}\nTEST 2: TEXT -> MULTI-IMAGE RETRIEVAL\n{'='*50}")
query_idx = 1
req_emb = text_embeddings[query_idx : query_idx + 1]
scores = (req_emb @ image_embeddings.T).squeeze().cpu().numpy()

for i, name in enumerate(image_names):
    print(f"Score: {scores[i]:.4f} -> Image: {name}")

# Test 3: Image to Multi-Image
print(f"\n{'='*50}\nTEST 3: IMAGE -> MULTI-IMAGE RETRIEVAL\n{'='*50}")
scores = (image_embeddings[0:1] @ image_embeddings.T).squeeze().cpu().numpy()

for i, name in enumerate(image_names):
    print(f"Score: {scores[i]:.4f} -> Image: {name}")